In [ ]:
import pandas as pd
import statistics
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
df = pd.read_csv("../data/mle_screening_dataset.csv")
df.head(5)

In [ ]:
print("question: ", df.iloc[0]['question'], "\nanswer: ", df.iloc[0]['answer'])

In [ ]:
df.shape

In [ ]:
df.isnull().sum().to_dict()

In [ ]:
df.duplicated().sum()

In [ ]:
df['question'].duplicated().sum()

In [ ]:
df['answer'].duplicated().sum()

In [ ]:
dup_counts = df.value_counts().reset_index(name="count")
dup_counts[dup_counts["count"] > 1].head(10)

In [ ]:
def normalize_question(q):
    """
    Produce a normalized key so near-duplicates map to the same string.
    """
    if not isinstance(q, str): return ''
    q = q.strip().lower()
    q = re.sub(r'\s+', ' ', q)
    q = re.sub(r'\s+([?.!,;:])', r'\1', q)
    q = re.sub(r'\?{2,}$', '?', q)
    return q

In [ ]:
def length_stats(series):
    """
    Compute token-length distribution stats for a Pandas Series.
    """
    lengths = series.astype(str).str.split().apply(len)
    return {
        "count": int(lengths.shape[0]),
        "min": int(lengths.min() if len(lengths) else 0),
        "p10": float(lengths.quantile(0.10)) if len(lengths) else 0.0,
        "median": float(lengths.median() if len(lengths) else 0),
        "p90": float(lengths.quantile(0.90)) if len(lengths) else 0.0,
        "max": int(lengths.max() if len(lengths) else 0),
        "mean": float(lengths.mean() if len(lengths) else 0.0),
        "std": float(lengths.std() if len(lengths) else 0.0,
        ),
    }

In [ ]:
def plot_hist(data, title, xlabel, out_path):
    """
    Plotting helper; Save the plots for visibility / generating reports
    """
    fig = plt.figure()
    plt.hist(data, bins=50)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Count')
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

In [ ]:
def top_ngrams(series, n=2, topk=25):
    """
    Word level top n-grams.
    Splits into word tokens, counts contiguous n-grams (default bigrams)
    """
    tokens = series.astype(str).str.lower().str.replace(r'[^a-z0-9\s]', ' ', regex=True).str.split()
    grams = Counter()
    for toks in tokens:
        for i in range(len(toks)-n+1):
            grams[tuple(toks[i:i+n])] += 1
    items = [{'ngram':' '.join(k), 'count':v} for k,v in grams.most_common(topk)]
    return pd.DataFrame(items)

In [ ]:
def detect_near_duplicates(texts, sample_size=2000, threshold=0.10):
    """
    Flags pairs with near duplicates using cosine distance 
    in TF-IDF space (0 = identical, 2 = opposite direction, but practically 0?1).
    Using hand-wavy threshold as 0.1, but can be tuned.
    """
    idxs = np.arange(len(texts))
    if len(idxs) > sample_size:
        rng = np.random.default_rng(42)
        idxs = rng.choice(idxs, size=sample_size, replace=False)
    subset = texts.iloc[idxs].tolist()
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=2)
    X = vec.fit_transform(subset)
    D = cosine_distances(X)
    pairs = []
    for i in range(len(subset)):
        for j in range(i+1, len(subset)):
            d = float(D[i, j])
            if d <= threshold:
                pairs.append((subset[i], subset[j], d))
    return pd.DataFrame(pairs, columns=['q1','q2','cosine_distance']).sort_values('cosine_distance')

In [ ]:
import re
df['q_norm'] = df['question'].apply(normalize_question)

In [ ]:
dup_counts = df.groupby('q_norm').size().sort_values(ascending=False)
n_unique = int(dup_counts.shape[0])
n_dupe_groups = int((dup_counts>1).sum())
top_dupes = dup_counts.head(20).reset_index().rename(columns={0:'count'})

In [ ]:
print(f"Total unique normalized questions: {n_unique:,}")
print(f"Groups with duplicates: {n_dupe_groups:,} "
      f"({n_dupe_groups / n_unique:.2%} of unique)")
print("\nTop duplicate questions:")
print(top_dupes.to_string(index=False))

In [ ]:
# Get length of questions and answers for potential insights
def stats(series): return series.astype(str).str.split().apply(len).tolist()
q_lengths = stats(df['question'])
a_lengths = stats(df['answer'])

In [ ]:
# Histogram plots plots
plot_hist(q_lengths, 'Question Length Distribution (tokens)', 'Question length', '../plots/question_len_hist.png')
plot_hist(a_lengths, 'Answer Length Distribution (tokens)', 'Answer length', '../plots/answer_len_hist.png')


In [ ]:
median_q_lengths = statistics.median(q_lengths)
print(f"Median of q_lengths: {median_q_lengths}")

median_a_lengths = statistics.median(a_lengths)
print(f"Median of a_lengths: {median_a_lengths}")

### Exploration of question and answer lengths:
- Questions are short and concise (median 8 tokens), which simplifies preprocessing and ensures queries remain focused.
- Answers, however, vary dramatically: while most are concise, some are very long, indicating the need for preprocessing strategies such as truncation, summarization, or chunking.

This asymmetry suggests that retrieval is a suitable approach: queries are short, but the system must handle variable-length answers robustly.

In [ ]:
# openings
first_two = df['q_norm'].str.split().apply(lambda x: ' '.join(x[:2]) if isinstance(x, list) and len(x)>=2 else '')
top_openings = first_two.value_counts().head(25).reset_index()
top_openings.columns = ['opening','count']

In [ ]:
print("\nTop 10 question openings:")
print(top_openings.head(10))

In [ ]:
from collections import Counter
bigrams = top_ngrams(df['question'], 2, 30)
trigrams = top_ngrams(df['question'], 3, 30)
print("\nMost common bigrams in questions:")
print(bigrams.head())
print("\nMost common trigrams in questions:")
print(trigrams.head())

In [ ]:
# near dupes
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
near_dupes = detect_near_duplicates(df['q_norm'], 2000, 0.1)
near_dupes.to_csv('../data/near_duplicate_questions_sampled.csv', index=False)
near_dupes.head()

In [ ]:
# outlier answers
a_len_series = df['answer'].astype(str).str.split().apply(len)
short_ans = df[a_len_series <= 5]
long_ans = df[a_len_series >= max(200, int(a_len_series.quantile(0.95)))]

In [ ]:
short_ans.head(3)

In [ ]:
long_ans.head(2)

From short answers, we observe rows where the answer is exactly or semantically similar to the question. We will have to remove such rows during pre-processing. This encourages us to compare answers for sematically similar questions

In [ ]:

total_rows = len(df)
unique_answers = df['answer'].astype(str).nunique()

# number of rows that are duplicates of another answer
duplicate_answer_rows = total_rows - unique_answers

# number of unique answers that are duplicated at least once
duplicated_answer_groups = df['answer'].astype(str).duplicated(keep=False).sum()

print(f"Total rows: {total_rows:,}")
print(f"Unique answers: {unique_answers:,}")
print(f"Rows with duplicate answers: {duplicate_answer_rows:,}")
print(f"Groups of answers duplicated at least once: {duplicated_answer_groups:,}")


In [ ]:
len(near_dupes)

Our dataset shows 594 rows with duplicate answers and at least 243 pairs of near-duplicate questions. These redundancies highlight the need for a deduplication step. We should normalize questions and merge answers for semantically similar entries, ensuring each unique medical query is represented once with a consolidated answer. This will improve training stability and retrieval quality.